# Sarcasm Detection — Colab Pro runner

Runs the whole project (TF-IDF baseline + BERT + RoBERTa, each **with** and **without** thread context) on the SARC dataset, then prints the comparison table and saves the results.

## Before you start
1. **Runtime → Change runtime type → GPU** (A100 or L4 if offered; a T4 works too, just slower).
2. Put **`train-balanced-sarcasm.csv`** somewhere in your Google Drive — you'll point to it in step 3. Get it from <https://www.kaggle.com/datasets/danofer/sarcasm>.

Then **Runtime → Run all**. With Colab Pro background execution it keeps running even if you close the tab.

In [ ]:
# 1. Which GPU did Colab give us?
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Get the code (the branch with all the project changes) and install dependencies.
!git clone -b gpu-cloud-setup https://github.com/jan-stochnialek/SarcasmDetection.git
%cd SarcasmDetection/project
!pip install -q -r requirements.txt

In [ ]:
# 3. Point the loader at your data file in Google Drive.
#    EDIT the path below to wherever you put train-balanced-sarcasm.csv.
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['SARC_CSV'] = '/content/drive/MyDrive/sarcasm/train-balanced-sarcasm.csv'
assert os.path.exists(os.environ['SARC_CSV']), 'CSV not found — fix the path above.'

# Alternative to Drive — download with the Kaggle API instead (also grabs comments.json
# for the official test split). Upload your kaggle.json first, then uncomment:
# !pip install -q kaggle && mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d danofer/sarcasm -p ../data/raw --unzip
# os.environ.pop('SARC_CSV', None)   # use the default data/raw path instead

In [ ]:
# 4. Pick a batch size that fits this GPU's memory. (bf16 turns on automatically on
#    A100/L4; a T4 falls back to fp16 — no action needed.)
import torch
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
batch = 64 if mem_gb >= 70 else 48 if mem_gb >= 38 else 32 if mem_gb >= 22 else 16
print(f'{torch.cuda.get_device_name(0)}  ~{mem_gb:.0f} GB  ->  BATCH_SIZE = {batch}')
!sed -i 's/^BATCH_SIZE = .*/BATCH_SIZE = {batch}/' settings.py

# On a slower GPU (T4), full data x 2 epochs takes a while. Uncomment these to use a
# 300k sample and a single epoch instead — much faster, results stay very close:
# !sed -i 's/^SAMPLE_SIZE = .*/SAMPLE_SIZE = 300000/' settings.py
# !sed -i 's/^EPOCHS = .*/EPOCHS = 1/' settings.py

In [ ]:
# 5. Sanity-check the data loads, then run everything. The log is mirrored to Drive
#    so you keep it even if the runtime recycles.
!python check_data.py
!python run_everything.py 2>&1 | tee /content/drive/MyDrive/sarcasm/run.log

In [ ]:
# 6. Re-print the table and copy results (scores + confusion-matrix PNGs) to Drive.
!python show_results.py
!mkdir -p /content/drive/MyDrive/sarcasm/results && cp -r ../results/* /content/drive/MyDrive/sarcasm/results/
print('Saved results to Drive: MyDrive/sarcasm/results/')